# Setup

In [1]:
%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format = "retina"

In [2]:
import os
import sys
from pprint import pprint

# so that mllm_shap can be imported without installing the package
sys.path.insert(0, os.path.abspath("../mllm_shap/src"))

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TQDM_DISABLE"] = "1"
os.environ["LOG_LEVEL"] = "INFO"

In [3]:
import numpy as np
import pandas as pd
import torch

np.random.seed(42)

device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
print(f"Using device: {device}")

Using device: mps


In [4]:
from mllm_shap.connectors import LiquidAudio, ModelConfig
from mllm_shap.connectors.enums import ModelHistoryTrackingMode, Role, SystemRolesSetup
from mllm_shap.connectors.filters import KeepAllTokens
from mllm_shap.shap import Explainer, McShapExplainer, PreciseShapExplainer
from mllm_shap.shap.monte_carlo import approximate_budget
from mllm_shap.shap.embeddings import MeanReducer
from mllm_shap.shap.enums import Mode
from mllm_shap.shap.normalizers import IdentityNormalizer
from mllm_shap.shap.similarity import CosineSimilarity
from mllm_shap.utils.jupyter import display_shap_colors_df

# Usage

Define LiquidAudio model (this call loads it up to the memory!).

Create compact explainer that will make initial call and then explain it using shapley values using Precise Formula.

In [5]:
model = LiquidAudio(
    device=device, history_tracking_mode=ModelHistoryTrackingMode.TEXT
)  # track and generate only text history
shap = PreciseShapExplainer(
    mode=Mode.STATIC,  # use static embeddings
    # use mean pooling to reduce token embeddings to single embedding per audio, default
    embedding_reducer=MeanReducer(),
    similarity_measure=CosineSimilarity(),  # use cosine similarity to compare embeddings, default
    normalizer=IdentityNormalizer(),  # no normalization
)
explainer_precise = Explainer(model=model, shap_explainer=shap)
explainer_mc = Explainer(
    model=model, shap_explainer=McShapExplainer(fraction=0.2)
)  # use Monte Carlo with 20% of all possibilities

W1108 11:47:15.708000 35148 torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Create new chat instance and assign it messages.

In [6]:
chat = model.get_new_chat(
    system_roles_setup=SystemRolesSetup.SYSTEM,  # calculate shapley values for all roles except system
    token_filter=KeepAllTokens(),  # keep all tokens for shapley values calculation
)

chat.new_turn(Role.USER)
chat.add_text("Who are you?")
chat.end_turn()

Let's have a look at chat representation:

In [7]:
pprint(chat.get_conversation())

[[ChatEntry(content_type=0, roles=[SYSTEM, SYSTEM, ..., SYSTEM, SYSTEM], content='<|im_start|>, user, \n, Who,  are,  you, ?, <|im_end|>, \n...', shap_values=None)]]


Representation is a list of list of ConversationEntry - so it can be accessed as  chat.get_conversation()[turn_number][message_number].{field}

Let's calculate shapley values for current conversation.

Verbose=True allows us to access history object (descried later). Generation kwargs allows to customize model interference - here we limit it to 4 tokens and change text_temperature from default 0.0 to 0.2, text_top_k from default 1 to 3. 

In [8]:
generation_kwargs = {"max_new_tokens": 4, "model_config": ModelConfig(text_temperature=0.2, text_top_k=3)}

result = explainer_precise(
    chat=chat,
    verbose=True,
    generation_kwargs=generation_kwargs,
    progress_bar=True,  # show progress bar during generation, default
)

2025-11-08 11:47:21,083 - mllm_shap.shap.compact - INFO - Generating full response from the model...
2025-11-08 11:47:21,671 - mllm_shap.shap.base.explainer - INFO - Number of tokens for explainability: 4 (up to 15 additional calls)


Calculating SHAP values:   0%|          | 0/15 [00:00<?, ?it/s]

2025-11-08 11:47:21,727 - mllm_shap.shap.base._generate_responses - WARNING - All text tokens were filtered out for mask 0, skipping.
2025-11-08 11:47:26,011 - mllm_shap.shap.base.explainer - INFO - Deduplicated 0/14 masks using existing cache.


DA


Note the warning - our chat consists only of user entries, therefore there will be mask that filters them out, keeping only some metadata. Such chat is irrelevant, therefore it gets skipped. Lets validate it based on history length:

In [9]:
len(result.history)

14

Result has now following fields available:

- full_chat - chat with base response (generated based on whole entry) with set cache and calculated shapley values
- source_chat - original chat feed to the explainer
- history - history of all chats used

Cache object stores actual shapley values as well as calculated embeddings and masks. They will be reused in next call regardless to the method, so for monte-carlo it is just larger sample, for precise it means some results might get excluded.

Let's first analyze history - it is a list of size equivalent to number of calls made for calculations + 1 (first entry, for base calculations, always None). In this case it is 14 entries not 15, because all False masks was excluded (we explain all tokens, it'd be empty text entry). Each entry is a tuple of following values:

- mask for that entry
- mash hash
- source chat with masked entry or None if corresponding mask was available in cache
- model response object

or None - when either corresponding mask was extracted from cache or it has risen an AllTextTokensFilteredOutError error.

Let's see all chats that were taken into account:

In [10]:
[c[2].decode_text() if c is not None else None for c in result.history]

['<|startoftext|><|im_start|>user\n?<|im_end|>\n',
 '<|startoftext|><|im_start|>user\n you<|im_end|>\n',
 '<|startoftext|><|im_start|>user\n you?<|im_end|>\n',
 '<|startoftext|><|im_start|>user\n are<|im_end|>\n',
 '<|startoftext|><|im_start|>user\n are?<|im_end|>\n',
 '<|startoftext|><|im_start|>user\n are you<|im_end|>\n',
 '<|startoftext|><|im_start|>user\n are you?<|im_end|>\n',
 '<|startoftext|><|im_start|>user\nWho<|im_end|>\n',
 '<|startoftext|><|im_start|>user\nWho?<|im_end|>\n',
 '<|startoftext|><|im_start|>user\nWho you<|im_end|>\n',
 '<|startoftext|><|im_start|>user\nWho you?<|im_end|>\n',
 '<|startoftext|><|im_start|>user\nWho are<|im_end|>\n',
 '<|startoftext|><|im_start|>user\nWho are?<|im_end|>\n',
 '<|startoftext|><|im_start|>user\nWho are you<|im_end|>\n']

Let's now analyze calculated shapley values.

In [11]:
explained_chat = result.full_chat

explained_chat_conversation = explained_chat.get_conversation()
pprint(explained_chat_conversation)

[[ChatEntry(content_type=0, roles=[SYSTEM, SYSTEM, ..., SYSTEM, SYSTEM], content='<|im_start|>, user, \n, Who,  are,  you, ?, <|im_end|>, \n...', shap_values=[nan, nan, ..., nan, nan])],
 [ChatEntry(content_type=0, roles=[SYSTEM, SYSTEM, ..., SYSTEM, SYSTEM], content='<|im_start|>, assistant, \n, I,  am,  an,  AI, <|im_end|>, \n...', shap_values=[nan, nan, ..., nan, nan])]]


Model was set to return just text tokens, so chat history has only text tokens. We can see that in the json representation inside ConversationEntry shap_values field is now populated. Nan values indicated that this token wasn't taken into calculation scope. As expected, we have 3 not-nan tokens. Let's see them.

In [12]:
user_entry = explained_chat_conversation[0][0]

display_shap_colors_df(
    pd.DataFrame(list(zip(user_entry.content, user_entry.shap_values)), columns=["Token", "Shapley Value"])
)

,Token,Shapley Value
0,<|im_start|>,nan
1,user,nan
2,,nan
3,Who,0.071777
4,are,0.034180
5,you,0.028320
6,?,-0.032959
7,<|im_end|>,nan
8,,nan


They clearly do not add up to 1, we use very inaccurate Monte Carlo approximation here without any later normalization.

We can asses number of calls required to achieving given importance:

In [13]:
approximate_budget(error_bound=0.1, confidence=0.9)

600

It's important to remember that result of this function is independent from our explainable tokens number - for that case we can have 2^4=16 calls anyway, so it is useless.

Let's create another turn to see how input significance will change:

In [14]:
explained_chat.new_turn(Role.USER)
explained_chat.add_text("Can you repeat?")
explained_chat.end_turn()

And again, let's explain it, this time we'll use Monte Carlo as number of possibilities will be too large. For that we'll need to remove existing cache, as for safety its usage is limited to same explainer instance. Due to a number of possibilities we won't track history this time.

In [15]:
del explained_chat.cache
result = explainer_mc(chat=explained_chat, verbose=False, generation_kwargs=generation_kwargs)

2025-11-08 11:47:26,623 - mllm_shap.shap.compact - INFO - Generating full response from the model...
2025-11-08 11:47:27,001 - mllm_shap.shap.base.explainer - INFO - Number of tokens for explainability: 12 (up to 4095 additional calls)


Calculating SHAP values:   0%|          | 0/819 [00:00<?, ?it/s]

2025-11-08 11:52:34,574 - mllm_shap.shap.base.explainer - INFO - Deduplicated 0/819 masks using existing cache.


DA


In [16]:
explained_chat = result.full_chat

explained_chat_conversation = explained_chat.get_conversation()
pprint(explained_chat_conversation)

[[ChatEntry(content_type=0, roles=[SYSTEM, SYSTEM, ..., SYSTEM, SYSTEM], content='<|im_start|>, user, \n, Who,  are,  you, ?, <|im_end|>, \n...', shap_values=[nan, nan, ..., nan, nan])],
 [ChatEntry(content_type=0, roles=[SYSTEM, SYSTEM, ..., SYSTEM, SYSTEM], content='<|im_start|>, assistant, \n, I,  am,  an,  AI, <|im_end|>, \n...', shap_values=[nan, nan, ..., nan, nan])],
 [ChatEntry(content_type=0, roles=[SYSTEM, SYSTEM, ..., SYSTEM, SYSTEM], content='<|im_start|>, user, \n, Can,  you,  repeat, ?, <|im_end|>, \n...', shap_values=[nan, nan, ..., nan, nan])],
 [ChatEntry(content_type=0, roles=[SYSTEM, SYSTEM, ..., SYSTEM, SYSTEM], content='<|im_start|>, assistant, \n, S, ure, ,,  I, <|im_end|>, \n...', shap_values=[nan, nan, ..., nan, nan])]]


In [17]:
dt = []
for i in (0, 1, 2):
    user_entry = explained_chat_conversation[i][0]
    df = pd.DataFrame(list(zip(user_entry.content, user_entry.shap_values)), columns=["Token", "Shapley Value"])
    df["Turn"] = i
    dt.append(df)

df = pd.DataFrame(pd.concat(dt), columns=["Token", "Shapley Value", "Turn"])
df = df.reset_index(drop=True)

display_shap_colors_df(df)

,Token,Shapley Value,Turn
0,<|im_start|>,nan,0
1,user,nan,0
2,,nan,0
3,Who,0.000000,0
4,are,0.026611,0
5,you,0.080078,0
6,?,0.053223,0
7,<|im_end|>,nan,0
8,,nan,0
9,<|im_start|>,nan,1


Lets display our chat:

In [18]:
for i, turn in enumerate(explained_chat_conversation):
    print(f"------ Turn {i}:")
    for entry in turn:
        entry.display()

------ Turn 0:
BY: USER, SYSTEM
TEXT CONTENT:
	<|im_start|> user 
	 Who  are  you ? <|im_end|> 
	
------ Turn 1:
BY: ASSISTANT, SYSTEM
TEXT CONTENT:
	<|im_start|> assistant 
	 I  am  an  AI <|im_end|> 
	
------ Turn 2:
BY: USER, SYSTEM
TEXT CONTENT:
	<|im_start|> user 
	 Can  you  repeat ? <|im_end|> 
	
------ Turn 3:
BY: ASSISTANT, SYSTEM
TEXT CONTENT:
	<|im_start|> assistant 
	 S ure ,  I <|im_end|> 
	


Let's note that one turn may contain SYSTEM messages even if it was USER / ASSISTANT turn, as all steering tokens are marked as SYSTEM.